In [2]:
# Main version, checked 17/12/2025
import csv
import os
from datasets import load_dataset
import spacy
from tqdm import tqdm
import torch
from tensormet.utils import DATA_DIR, select_gpu

# ---- helpers to stream just the text field ----

def gen_texts(dataset, dataset_column_name):
    for ex in dataset:
        # each ex is a dict like {"text": "...", ...}
        txt = ex.get(dataset_column_name)
        if txt:
            # let spaCy handle casing etc.; keep original text
            yield txt


def extract_vectors(doc, start_sent_id):
    """
    doc: spaCy Doc
    start_sent_id: int -> the id to assign to the FIRST sentence in this doc
    returns:
        rows: list of dicts, one per extracted vector
        count: how many vectors we added
        next_sent_id: the next free sentence id after this doc
    """

    S = doc.vocab.strings
    DEP_NSUBJ = S["nsubj"]
    DEP_OBJ = S["dobj"]
    DEP_OBL = S["obl"]
    ROOT_VERB = S["VERB"]

    rows = []
    vec_count = 0
    sent_id = start_sent_id

    for sent in doc.sents:
        root = sent.root
        # check that the root POS is a verb
        if root.pos != ROOT_VERB:  # equivalent to: if root.pos_ != "VERB"
            sent_id += 1
            continue

        nsubj = obj = obj2 = obl = "~"
        filled = 0  # how many slots filled

        for child in root.children:
            d = child.dep
            if d == DEP_NSUBJ and nsubj == "~":
                nsubj = child.lemma_
                filled += 1
            elif d == DEP_OBJ:
                if obj == "~":
                    obj = child.lemma_
                    filled += 1
                elif obj2 == "~":
                    obj2 = child.lemma_
                    filled += 1
            elif d == DEP_OBL and obl == "~":
                obl = child.lemma_
                filled += 1

            if filled == 4:
                break

        root_lemma = root.lemma_

        # build the row for CSV
        vector_tuple = (root_lemma, nsubj, obj, obj2, obl)

        rows.append({
            "sent_id": sent_id,
            # "root_lemma": root_lemma,
            # "nsubj": nsubj,
            # "obj": obj,
            # "obj2": obj2,
            # "obl": obl,
            "vector": repr(vector_tuple),     ### NEW: store full tuple in one field
            "sentence_text": sent.text        ### NEW: store original sentence text
        })

        vec_count += 1
        sent_id += 1  # advance to next sentence id

    return rows, vec_count, sent_id


def pos_tag(target_vectors=10000,
            save_every=1000,
            output_path=DATA_DIR/"vectors/fineweb_english_vectors.csv",
            dataset_path="HuggingFaceFW/fineweb",
            dataset_config="CC-MAIN-2025-26",
            dataset_column_name="text",
            spacy_model="en_core_web_md"):
    from datasets import load_dataset
    nlp = spacy.load(spacy_model, disable=["ner", "textcat"])
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # --- dataset stream ---
    ds = load_dataset(dataset_path, dataset_config,
                      split="train", streaming=True)
    ds = ds.shuffle(seed=7, buffer_size=10_000)

    # init CSV once (now with extra columns)
    if not os.path.exists(output_path):
        with open(output_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow([
                "sent_id",          ### NEW
                # "root_lemma",
                # "nsubj",
                # "obj",
                # "obj2",
                # "obl",
                "vector",           ### NEW
                "sentence_text"     ### NEW
            ])

    texts = gen_texts(ds, dataset_column_name)
    buffer = []
    vector_count = 0
    last_checkpoint_at = 0
    global_sent_id = 0          ### NEW: running sentence index

    try:
        # nlp.pipe yields Doc objects
        for doc in nlp.pipe(texts, batch_size=1000, n_process=50):
            rows, count, global_sent_id = extract_vectors(doc, global_sent_id)
            if count == 0:
                continue

            # append dict rows to buffer, but CSV writer wants flat lists
            for r in rows:

                buffer.append([
                    r["sent_id"],
                    # r["root_lemma"],
                    # r["nsubj"],
                    # r["obj"],
                    # r["obj2"],
                    # r["obl"],
                    r["vector"],
                    r["sentence_text"]
                ])

            vector_count += count
            if vector_count - last_checkpoint_at >= save_every:
                with open(output_path, "a", newline="", encoding="utf-8") as f:
                    w = csv.writer(f)
                    w.writerows(buffer)
                buffer.clear()
                last_checkpoint_at = vector_count

            if vector_count >= target_vectors:
                break

    except KeyboardInterrupt:
        pass
    finally:
        if buffer:
            with open(output_path, "a", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerows(buffer)

    print(f"Vectors written: {vector_count} (file: {output_path})")




<>:53: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:57: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:60: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:63: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:53: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:57: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:60: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:63: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
/tmp/ipykernel_3950259/2110066725.py:53: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if d == DEP_NSUBJ and nsubj is "~":
/tmp/ipykernel_3950259/2110066725.py:57: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if obj is "~":
/tmp/ipykernel_3950259/2110066725.py:60: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif obj2 is "~":
/tmp/ipykernel_3950259/2110066725.py:63: SyntaxWarning: "is" with 'str' literal. Did you mean "==

In [3]:
nlp = spacy.load("en_core_web_md", disable=["ner", "textcat"])

In [7]:
doc = nlp("The footballer kicks the ball. They give him an award. There is nothing to be done. All hail the queen! I'm eating. I'm eating biscuits. I don't know")


In [8]:
doc

The footballer kicks the ball. They give him an award. There is nothing to be done. All hail the queen! I'm eating. I'm eating biscuits. I don't know

In [10]:
for sent in doc.sents:
    root = sent.root
    print(root.lemma_)
    for child in root.children:
        print(child, child.dep_)

kick
footballer nsubj
ball dobj
. punct
give
They nsubj
him dative
award dobj
. punct
be
There expl
nothing attr
. punct
hail
All nsubj
queen dobj
! punct
eat
I nsubj
'm aux
. punct
eat
I nsubj
'm aux
biscuits dobj
. punct
know
I nsubj
do aux
n't neg


In [11]:
from similarity import load_eval_sentences_cached
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")
dataset = "fineweb-en"
random_state = 1

sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset=dataset,
                                             seed=random_state,
                                             n_samples=10_000,
                                             )

In [12]:

from tucker_tensor import TuckerDecomposition
tensor = TuckerDecomposition.load_from_disk(
    dataset="preOOV-fineweb-en",
    method="siiShifted",
    dims=2000,
    rank=150,
    iterations=500
)

In [15]:
from similarity import ensure_vocab
clean_sample = ensure_vocab(tensor, sentence_sample)

100%|██████████| 10000/10000 [00:00<00:00, 569692.49it/s]


In [16]:
clean_sample

[('look', 'committee', '~'),
 ('reduce', 'cup', '~'),
 ('limit', 'food', 'effectiveness'),
 ('comment', 'Digger', '~'),
 ('add', 'it', '~'),
 ('create', '~', 'atmosphere'),
 ('empower', '~', 'organization'),
 ('see', 'police', 'I'),
 ('evolve', '~', '~'),
 ('leave', '~', 'student'),
 ('tweet', '~', '~'),
 ('wave', '~', 'head'),
 ('guide', '~', 'you'),
 ('mean', 'one', '~'),
 ('provide', 'partner', 'code'),
 ('caution', '~', '~'),
 ('retire', '~', '~'),
 ('help', 'platform', 'Service'),
 ('symbolize', 'dream', 'effort'),
 ('~', 'they', '~'),
 ('make', 'they', 'everything'),
 ('enter', 'you', '~'),
 ('emphasize', 'source', 'importance'),
 ('test', 'we', '~'),
 ('offer', 'rental', 'convenience'),
 ('unpack', 'they', '~'),
 ('symbolize', '~', 'balance'),
 ('cause', '~', '~'),
 ('s', 'doctor', '~'),
 ('jump', 'detail', '~'),
 ('spotlight', '~', 'motivation'),
 ('make', '~', '~'),
 ('help', '~', 'disorder'),
 ('perform', 'Texas', '~'),
 ('develop', 'approach', 'cooperation'),
 ('impede', 'co

In [17]:
sentence_sample

[('look', 'committee', '~'),
 ('reduce', 'cup', '~'),
 ('limit', 'food', 'effectiveness'),
 ('comment', 'Digger', '~'),
 ('add', 'it', 'gloss'),
 ('create', 'freshener', 'atmosphere'),
 ('empower', 'Hyperwallet', 'organization'),
 ('see', 'police', 'I'),
 ('evolve', 'Shifts', '~'),
 ('leave', 'deficit', 'student'),
 ('tweet', 'Anantha', '~'),
 ('wave', 'foliage', 'head'),
 ('guide', 'Specialists', 'you'),
 ('mean', 'one', '~'),
 ('provide', 'partner', 'code'),
 ('caution', 'Ayodele', 'Nigerians'),
 ('retire', 'Rogers', '~'),
 ('help', 'platform', 'Service'),
 ('symbolize', 'dream', 'effort'),
 ('budge', 'they', '~'),
 ('make', 'they', 'everything'),
 ('enter', 'you', 'kingdom'),
 ('emphasize', 'source', 'importance'),
 ('test', 'we', 'Tutorial'),
 ('offer', 'rental', 'convenience'),
 ('unpack', 'they', 'intersection'),
 ('symbolize', 'myth', 'balance'),
 ('cause', 'Blast', 'discoloration'),
 ('s', 'doctor', '~'),
 ('jump', 'detail', '~'),
 ('spotlight', 'huberman', 'motivation'),
 ('ma

In [18]:
"Digger" in tensor.vocab["s2i"]

True

In [19]:
import torch
from utils import DATA_DIR
tensor = torch.load(DATA_DIR/"tensors/lassy/populated/sc_1000.pt")

In [21]:
tensor.shape

torch.Size([1000, 1000, 1000])

In [23]:
import pickle

with open(DATA_DIR/"tensors/fineweb-en/vocabularies/1000.pkl", "rb") as f:
    vocab = pickle.load(f)

In [25]:
vocab["vocab_s"]

['~',
 'you',
 'I',
 'we',
 'it',
 'they',
 'he',
 'this',
 'she',
 'that',
 'company',
 'team',
 'people',
 'one',
 'player',
 'system',
 'user',
 'study',
 'student',
 'some',
 'what',
 'service',
 'business',
 'approach',
 'platform',
 'game',
 'design',
 'these',
 'casino',
 'program',
 'process',
 'feature',
 'report',
 'People',
 'article',
 'many',
 'member',
 'tool',
 'customer',
 'group',
 'individual',
 'government',
 'site',
 'number',
 'research',
 'man',
 'project',
 'technology',
 'event',
 'thing',
 'work',
 'all',
 'organization',
 'app',
 'type',
 'who',
 'expert',
 'woman',
 'everyone',
 'website',
 'experience',
 'product',
 '%',
 'child',
 'model',
 'step',
 'option',
 'method',
 'researcher',
 'family',
 'result',
 'those',
 'solution',
 'guide',
 'not',
 'plan',
 'person',
 'change',
 'benefit',
 'book',
 'other',
 'brand',
 'story',
 'time',
 'price',
 'strategy',
 'God',
 'patient',
 'client',
 'official',
 'participant',
 'leader',
 'professional',
 'factor',
 

In [28]:
"~" in vocab["vocab_v"]

False